In [9]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

def x_y_mnist(num_classes, input_shape):    
    (x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()
    
    x_train = x_train.astype("float32") / 255
    x_test = x_test.astype("float32") / 255
    x_train = np.expand_dims(x_train, -1)
    x_test = np.expand_dims(x_test, -1)
    print("x_train shape:", x_train.shape)
    print(x_train.shape[0], "train samples")
    print(x_test.shape[0], "test samples")
    
    y_train = keras.utils.to_categorical(y_train, num_classes)
    y_test = keras.utils.to_categorical(y_test, num_classes)
    return x_train, y_train, x_test, y_test


In [10]:
def train_keras(x_train, y_train, num_classes, input_shape):
    batch_size, epochs = 128, 5
    
    model = keras.Sequential(
        [
            keras.Input(shape=input_shape),
            layers.Conv2D(32, kernel_size=(3, 3), activation="relu"),
            layers.MaxPooling2D(pool_size=(2, 2)),
            layers.Conv2D(64, kernel_size=(3, 3), activation="relu"),
            layers.MaxPooling2D(pool_size=(2, 2)),
            layers.Flatten(),
            layers.Dropout(0.5),
            layers.Dense(num_classes, activation="softmax"),
        ]
    )
    
    model.summary()
    model.compile(loss="categorical_crossentropy", optimizer="adam", metrics=["accuracy"])
    model.fit(x_train, y_train, batch_size=batch_size, epochs=epochs, validation_split=0.1)

    return model

In [3]:
import tensorflow as tf
import tf2onnx

def keras_to_onnx(model, path):
    model.export(path)
    !python -m tf2onnx.convert --saved-model "{path}" --output "{path}.onnx" --opset 13

In [4]:
import time
import onnxruntime as ort
import tensorflow as tf

def benchmark_model(model, onnx_session, x_data, batch_size, onnx_input_name = None, onnx_output_name = None):
    x_subset = x_data[:1000]  
    num_batches = len(x_subset) // batch_size

    if onnx_input_name:
        onnx_session.run([onnx_output_name], {onnx_input_name: x_subset[:batch_size].astype(np.float32)})
    else:
        model.predict(x_subset[:batch_size], verbose=0)

    start = time.time()
    for i in range(num_batches):
        batch = x_subset[i*batch_size:(i+1)*batch_size]
        if onnx_input_name:
            onnx_session.run([onnx_output_name], {onnx_input_name: batch.astype(np.float32)})
        else:
            model.predict(batch, verbose=0)
    end = time.time()

    avg_time = (end - start) / num_batches
    return avg_time

def models_speed(model, onnx_session, x_test, onnx_input_name, onnx_output_name):
    for bs in [1, 8, 32, 128]:
        keras_time = benchmark_model(model, onnx_session, x_test, bs)
        onnx_time = benchmark_model(model, onnx_session, x_test, bs, onnx_input_name, onnx_output_name)
        print(f"Batch size {bs:3d} | Keras: {keras_time:.6f}s | ONNX: {onnx_time:.6f}s")


In [13]:
def compare_accuracy_softpreds(model, onnx_session, onnx_input_name, onnx_output_name):
    keras_preds = model.predict(x_test, verbose=0)
    keras_acc = np.mean(np.argmax(keras_preds, axis=1) == np.argmax(y_test, axis=1))

    onnx_preds = []
    for i in range(0, len(x_test), 128):
        batch = x_test[i:i+128].astype(np.float32)
        preds = onnx_session.run([onnx_output_name], {onnx_input_name: batch})[0]
        onnx_preds.append(preds)
    onnx_preds = np.vstack(onnx_preds)
    onnx_acc = np.mean(np.argmax(onnx_preds, axis=1) == np.argmax(y_test, axis=1))

    print(f"\nKeras test accuracy: {keras_acc:.4f}")
    print(f"ONNX test accuracy:  {onnx_acc:.4f}")
    
    idxs = np.random.choice(len(x_test), 5, replace=False)
    for i in idxs:
        keras_p = keras_preds[i]
        onnx_p = onnx_preds[i]
        diff = np.abs(keras_p - onnx_p).max()
        print(f"Sample {i}: max abs diff = {diff:.8f}")

In [6]:
from onnxruntime.quantization import quantize_static, CalibrationDataReader, QuantType
import onnx

class DataReader(CalibrationDataReader):
    def __init__(self, data):
        self.data = data
        self.enum_data = None

    def get_next(self):
        if self.enum_data is None:
            self.enum_data = iter([{"keras_tensor": self.data.astype(np.float32)}])
        return next(self.enum_data, None)

def quantize(onnx_model_path, quantized_model_path, x_test):
    calibration_data = x_test[:100]
    
    dr = DataReader(calibration_data)
    
    quantize_static(
        model_input=onnx_model_path,
        model_output=quantized_model_path,
        calibration_data_reader=dr,
        quant_format="QDQ",
        weight_type=QuantType.QInt8,
        activation_type=QuantType.QInt8
    )

In [11]:
num_classes, input_shape = 10, (28, 28, 1)

x_train, y_train, x_test, y_test = x_y_mnist(num_classes, input_shape)
mnist_model = train_keras(x_train, y_train, num_classes, input_shape)

x_train shape: (60000, 28, 28, 1)
60000 train samples
10000 test samples


I0000 00:00:1760363528.539952  186364 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2025-10-13 15:52:08.542014: W tensorflow/core/common_runtime/gpu/gpu_device.cc:2343] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 26, 26, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 13, 13, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 11, 11, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 5, 5, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 1600)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1600)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 10)             │        16,010 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 34,826 (136.04 KB)

 Trainable params: 34,826 (136.04 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/5
422/422 ━━━━━━━━━━━━━━━━━━━━ 8s 17ms/step - accuracy: 0.8872 - loss: 0.3651 - val_accuracy: 0.9765 - val_loss: 0.0876
Epoch 2/5
422/422 ━━━━━━━━━━━━━━━━━━━━ 7s 17ms/step - accuracy: 0.9644 - loss: 0.1151 - val_accuracy: 0.9833 - val_loss: 0.0601
Epoch 3/5
422/422 ━━━━━━━━━━━━━━━━━━━━ 7s 16ms/step - accuracy: 0.9738 - loss: 0.0844 - val_accuracy: 0.9853 - val_loss: 0.0507
Epoch 4/5
422/422 ━━━━━━━━━━━━━━━━━━━━ 6s 14ms/step - accuracy: 0.9773 - loss: 0.0721 - val_accuracy: 0.9873 - val_loss: 0.0438
Epoch 5/5
422/422 ━━━━━━━━━━━━━━━━━━━━ 6s 15ms/step - accuracy: 0.9801 - loss: 0.0632 - val_accuracy: 0.9897 - val_loss: 0.0391


In [ ]:
keras_to_onnx(mnist_model, "mnist_model")

In [14]:
mnist_onnx_session = ort.InferenceSession("mnist_model.onnx", providers=["CPUExecutionProvider"])
mnist_onnx_input_name, mnist_onnx_output_name = mnist_onnx_session.get_inputs()[0].name, mnist_onnx_session.get_outputs()[0].name

models_speed(mnist_model, mnist_onnx_session, x_test, mnist_onnx_input_name, mnist_onnx_output_name)
compare_accuracy_softpreds(mnist_model, mnist_onnx_session, mnist_onnx_input_name, mnist_onnx_output_name)

Batch size   1 | Keras: 0.048350s | ONNX: 0.000052s
Batch size   8 | Keras: 0.048354s | ONNX: 0.000128s
Batch size  32 | Keras: 0.044276s | ONNX: 0.000361s
Batch size 128 | Keras: 0.051341s | ONNX: 0.001086s

Keras test accuracy: 0.9869
ONNX test accuracy:  0.9883
Sample 5067: max abs diff = 0.13787299
Sample 7108: max abs diff = 0.00005898
Sample 309: max abs diff = 0.00001991
Sample 3488: max abs diff = 0.00003302
Sample 2276: max abs diff = 0.34353089


In [15]:
quantize("mnist_model.onnx", "mnist_model_quantized.onnx", x_test)

mnist_quantized_onnx_session = ort.InferenceSession("mnist_quantized_model.onnx", providers=["CPUExecutionProvider"])
mnist_quantized_onnx_input_name, mnist_quantized_onnx_output_name = mnist_quantized_onnx_session.get_inputs()[0].name, mnist_quantized_onnx_session.get_outputs()[0].name

models_speed(mnist_model, mnist_quantized_onnx_session, x_test, mnist_quantized_onnx_input_name, mnist_quantized_onnx_output_name)
compare_accuracy_softpreds(mnist_model, mnist_quantized_onnx_session, mnist_quantized_onnx_input_name, mnist_quantized_onnx_output_name)

ValueError: Required inputs (['keras_tensor_8']) are missing from input feed (['keras_tensor']).


Keras test accuracy: 0.9914
ONNX test accuracy:  0.9914
Sample 6224: max abs diff = 0.00000000
Sample 848: max abs diff = 0.00000012
Sample 6052: max abs diff = 0.00000000
Sample 5574: max abs diff = 0.00000012
Sample 607: max abs diff = 0.00000012


In [23]:
onnx_quantized_session = ort.InferenceSession("model_quantized.onnx", providers=["CPUExecutionProvider"])

compare_accuracy_softpreds(onnx_quantized_session, model)


Keras test accuracy: 0.9914
ONNX test accuracy:  0.9915
Sample 7494: max abs diff = 0.00390923
Sample 5808: max abs diff = 0.00390732
Sample 8857: max abs diff = 0.00391090
Sample 9600: max abs diff = 0.00388253
Sample 2605: max abs diff = 0.00391996


In [26]:
import os

orig_size = os.path.getsize("model.onnx") / (1024 * 1024)
quant_size = os.path.getsize("model_quantized.onnx") / (1024 * 1024)

print(f"Non-quantized model size : {orig_size:.2f} MB")
print(f"Quantized model size : {quant_size:.2f} MB")

Non-quantized model size : 0.14 MB
Quantized model size : 0.05 MB
